## 读取数据

In [1]:
data_path="../../data/ultrafineweb-en-part-0036-of-2048.parquet"

In [2]:
import pandas as pd

In [3]:
df=pd.read_parquet(data_path)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 566016 entries, 0 to 566015
Data columns (total 3 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   content  566016 non-null  object
 1   score    566016 non-null  object
 2   source   566016 non-null  object
dtypes: object(3)
memory usage: 13.0+ MB


In [5]:
from openai import OpenAI

In [6]:
import rich

In [7]:
base_url="http://localhost:8000/v1"
model_name="Qwen3-8B"
api_key="dummy_key"

In [8]:
client = OpenAI(
    base_url=base_url,
    api_key=api_key
)

In [9]:
def save_demo(response:str,file_name:str="demo.txt")->None:
    with open(file_name,"w") as f:
        f.write(response)

## tag_annotation

In [10]:
tag_annotation_prompt=""
with open("../../src/gem/prompts/tag_annotation.md","r") as f:
    tag_annotation_prompt=f.read()

In [11]:
def call_llm(prompt:str)->str:
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role":"user","content":prompt}],
        temperature=0.7,
        max_tokens=40960,
        top_p=0.95
    )
    return response.choices[0].message.content.rsplit("</think>",1)[-1].lstrip()

In [12]:
def tag_annotation(seed:int)->str:
    prompt=tag_annotation_prompt.replace("{text}",df.loc[seed,"content"])
    return call_llm(prompt)

In [13]:
import re
from typing import Optional
from pydantic import BaseModel

class TagAnnotation(BaseModel):
    """输出字符串的结构化数据类，对应所有提取字段"""
    multi_step: bool  # 多步标识，布尔型
    summary: str      # 摘要，字符串
    domain: str       # 领域，字符串（支持多值逗号分隔）
    platform: str     # 平台，字符串
    task: str         # 任务，字符串

In [14]:
def extract_tag_annotation(response: str) -> TagAnnotation:
    """
    解析输出字符串，提取字段并实例化TagAnnotation对象
    :param response: 待解析的标签格式字符串
    :return: TagAnnotation实例，可通过.属性访问所有字段
    """
    # 正则匹配函数：封装通用匹配逻辑，处理空白、默认值
    def _match_tag(pattern: str, default: Optional[str] = "") -> str:
        match = re.search(pattern, response, re.DOTALL)  # re.DOTALL让.匹配换行符
        return match.group(1).strip() if match else default

    # 逐个匹配各标签的内容，正则模式为<标签>(.*?)</标签>（非贪婪匹配）
    multi_step_str = _match_tag(r'<multi_step>(.*?)</multi_step>', default='False')
    summary = _match_tag(r'<summary>(.*?)</summary>')
    domain = _match_tag(r'<domain>(.*?)</domain>')
    platform = _match_tag(r'<platform>(.*?)</platform>')
    task = _match_tag(r'<task>(.*?)</task>')

    multi_step = multi_step_str.lower() == 'true'

    # 实例化数据类并返回
    return TagAnnotation(
        multi_step=multi_step,
        summary=summary,
        domain=domain,
        platform=platform,
        task=task
    )

In [15]:
response=tag_annotation(6)

In [16]:
rich.print(extract_tag_annotation(response))

TagAnnotation(multi_step=False, summary='', domain='', platform='', task='')

In [17]:
rich.print(response)

<multi_step>False</multi_step>

In [18]:
# for i in range(100):
#     response=tag_annotation(i)
#     ta=extract_tag_annotation(response)
#     if ta.multi_step:
#         rich.print(i)
#         rich.print(ta)
#         break

In [19]:
import json

ta=extract_tag_annotation(response)
save_demo(json.dumps(ta.model_dump(),ensure_ascii=False,indent=4),"tag_annotation_demo.txt")

## workflow_and_tool_discovery

In [20]:
workflow_and_tool_discovery_prompt=""
with open("../../src/gem/prompts/workflow_and_tool_discovery.md","r") as f:
    workflow_and_tool_discovery_prompt=f.read()

In [21]:
def workflow_and_tool_discovery(seed:int)->str:
    prompt=workflow_and_tool_discovery_prompt.replace("{text}",df.loc[seed,"content"])
    return call_llm(prompt)

In [22]:
import json

In [23]:
from pydantic import BaseModel, field_validator

class Workflow(BaseModel):
    description: str
    steps: str
    execution_graph: str
    actions: list[dict]
    tools: list[dict]

    @field_validator('steps')
    @classmethod
    def fix_newlines(cls, v: str) -> str:
        # 把所有的双换行或多换行统一缩减为单换行
        # 同时处理掉可能存在的 \\n 这种字面量字符串
        v = v.replace('\\n', '\n') 
        return re.sub(r'\n\s*\n', '\n', v).strip()

In [24]:
import dirtyjson
import json

# ------------------- 核心提取函数：extract_workflows -------------------
def extract_workflows(response: str) -> list[Workflow]:
    """
    从响应字符串中提取所有<workflow>块，解析为List[Workflow]
    :param response: 包含一个/多个<workflow>的原始响应字符串
    :return: Workflow对象列表，无匹配时返回空列表
    """
    def _match_tag(workflow_block: str, tag: str, default: str = "") -> str:
        """
        从单个workflow块中匹配指定子标签的内容，处理空白和缺失
        :param workflow_block: 单个<workflow>内的字符串
        :param tag: 子标签名（如description/steps）
        :param default: 匹配不到的默认值
        :return: 标签内的内容（已去前后空白）
        """
        pattern = re.compile(rf'<{tag}>(.*?)</{tag}>', re.DOTALL)  # re.DOTALL适配换行
        match = pattern.search(workflow_block)
        return match.group(1).strip() if match else default
    
    
    def _safe_json_loads(json_str: str, default: Optional[any] = None) -> any:
        if default is None: 
            default = []
        if not json_str: 
            return default
        try:
            # dirtyjson 比原生 json.loads 容错率高得多
            return dirtyjson.loads(json_str)
        except Exception:
            # 如果还是不行，尝试用正则强行补齐简单的括号错误（可选）
            return default
            
    workflows: List[Workflow] = []
    # 第一步：匹配所有<workflow>...</workflow>块（非贪婪匹配，获取独立的工作流字符串）
    workflow_pattern = re.compile(r'<workflow>(.*?)</workflow>', re.DOTALL | re.MULTILINE)
    workflow_blocks = workflow_pattern.findall(response)

    # 第二步：遍历每个workflow块，逐个解析内部子标签
    for block in workflow_blocks:
        # 匹配各子标签的原始字符串
        desc = _match_tag(block, "description")
        steps = _match_tag(block, "steps")
        exec_graph = _match_tag(block, "execution_graph")
        actions_str = _match_tag(block, "actions")
        tools_str = _match_tag(block, "tools")

        # 解析JSON格式的actions和tools（安全解析，失败返回空列表）
        actions = _safe_json_loads(actions_str)
        tools = _safe_json_loads(tools_str)

        # 实例化Workflow并加入列表
        workflow = Workflow(
            description=desc,
            steps=steps,
            execution_graph=exec_graph,
            actions=actions,
            tools=tools
        )
        workflows.append(workflow)

    return workflows

In [25]:
response=workflow_and_tool_discovery(6)

In [26]:
rich.print(response)

<workflow>
<description>No valid workflow found in the provided text. The content appears to be a mix of academic questions 
and an advertisement for an essay writing service, which does not contain structured workflow steps or executable 
processes.</description>
<steps></steps>
<execution_graph></execution_graph>
<actions>[]</actions>
<tools>[]</tools>
</workflow>

In [27]:
wls=extract_workflows(response)

In [28]:
rich.print(wls)

[
    Workflow(
        description='No valid workflow found in the provided text. The content appears to be a mix of academic 
questions and an advertisement for an essay writing service, which does not contain structured workflow steps or 
executable processes.',
        steps='',
        execution_graph='',
        actions=[],
        tools=[]
    )
]

In [29]:
json_data = json.dumps([wf.model_dump() for wf in wls], ensure_ascii=False, indent=4)
rich.print(json_data)

[
    {
        "description": "No valid workflow found in the provided text. The content appears to be a mix of academic 
questions and an advertisement for an essay writing service, which does not contain structured workflow steps or 
executable processes.",
        "steps": "",
        "execution_graph": "",
        "actions": [],
        "tools": []
    }
]

In [30]:
save_demo(json_data,"workflows_demo.txt")

In [31]:
# with open("workflows_demo.txt", "r") as f:
#     raw_data = json.load(f) 

# workflows = [Workflow.model_validate(item) for item in raw_data]
# rich.print(workflows)

## trajectory_generation

In [32]:
from pydantic import BaseModel, Field
from typing import List, Optional, Union, Dict, Any
import json

class ToolDefinition(BaseModel):
    """对应 <toolsets> 中的单个工具定义"""
    name: str
    description: str
    parameters: Dict[str, Any]
    
class ToolCall(BaseModel):
    """存储 <func> 标签内的工具调用信息"""
    name: str
    arguments: dict

class Message(BaseModel):
    """单条消息结构"""
    role: str  # 'user', 'assistant', 'tool'
    content: str
    # 只有 assistant 角色可能包含 tool_calls
    tool_calls: Optional[List[ToolCall]] = None

class Dialogue(BaseModel):
    """完整的对话"""
    system_prompt: str
    conversation: List[Message]

class Trajectory(BaseModel):
    """完整的轨迹"""
    toolsets: List[ToolDefinition]
    system_prompt: str
    conversation: List[Message]

In [33]:
import re

def extract_dialogue(response: str) -> Optional[Dialogue]:
    # 1. 提取 System Prompt
    system_match = re.search(r'<system>(.*?)</system>', response, re.DOTALL)
    system_prompt = system_match.group(1).strip() if system_match else ""

    # 2. 提取所有对话轮次 (user, assistant, tool)
    # 我们使用正则匹配所有合法的标签对
    # 注意：这里不处理嵌套，先按顺序抓取所有顶级标签
    tags_pattern = re.compile(r'<(user|assistant|tool)>(.*?)</\1>', re.DOTALL)
    all_turns = tags_pattern.findall(response)

    conversation = []

    for role, content in all_turns:
        content = content.strip()
        tool_calls = []

        # 3. 如果是 assistant，进一步解析内部的 <func> 标签
        if role == 'assistant':
            func_pattern = re.compile(r'<func>(.*?)</func>', re.DOTALL)
            funcs = func_pattern.findall(content)
            
            for f_json in funcs:
                try:
                    # 清洗 JSON 字符串（处理模型可能多出的换行）
                    clean_json = re.sub(r'[\x00-\x1F\x7F]', '', f_json.strip())
                    f_data = json.loads(clean_json)
                    tool_calls.append(ToolCall(
                        name=f_data.get("name", ""),
                        arguments=f_data.get("arguments", {})
                    ))
                except json.JSONDecodeError:
                    continue # 或者记录解析失败
            
            # 移除 content 中的 <func> 部分，只保留纯文本回复（可选）
            content = func_pattern.sub('', content).strip()

        conversation.append(Message(
            role=role,
            content=content,
            tool_calls=tool_calls if tool_calls else None
        ))

    return Dialogue(
        system_prompt=system_prompt,
        conversation=conversation
    )

In [34]:
trajectory_generation_prompt=""
with open("../../src/gem/prompts/trajectory_generation.md","r") as f:
    trajectory_generation_prompt=f.read()

In [35]:
def trajectory_generation(workflow:Workflow)->str:
    steps:str=workflow.steps
    tools:list[dict]=workflow.tools
    tools_str=json.dumps(tools,ensure_ascii=False)
    prompt = (
        trajectory_generation_prompt
        .replace("{candidate_tools}", tools_str)
        .replace("{current_task}", steps)
    )
    return call_llm(prompt)

In [36]:
response=trajectory_generation(wls[0])

In [37]:
rich.print(response)

<system>
You are an AI assistant managing a library system. Here are the domain-specific rules:
- Users can only borrow books that are currently available (status: "available").
- Borrowing is limited to 3 books per user.
- Fines apply for overdue returns (50 cents per day).
- Users must provide valid ID to borrow books.
- Book reservations are allowed only if the book is on hold.
</system>

<user>
I want to borrow "The Great Gatsby" and "1984".
</user>
<assistant>
<func>{"name":"check_book_availability", "arguments": {"book_titles": ["The Great Gatsby", "1984"]}}</func>
</assistant>
<tool>
{"status": "success", "availability": {"The Great Gatsby": "available", "1984": "checked out"}}
</tool>
<assistant>
"The Great Gatsby" is available for borrowing, but "1984" is currently checked out. Would you like to reserve 
"1984" or choose a different book?
</assistant>
<user>
Reserve "1984" for me.
</user>
<assistant>
<func>{"name":"reserve_book", "arguments": {"book_title": "1984", "user_id": "U12345"}}</func>
</assistant>
<tool>
{"status": "success", "message": "Reservation request for '1984' has been processed. You will be notified when it 
becomes available."}
</tool>
<assistant>
Your reservation for "1984" is confirmed. You can borrow up to 3 books at a time. Would you like to proceed with 
borrowing "The Great Gatsby" now?
</assistant>
<user>
Yes, but I need to return "The Alchemist" first. How long has it been overdue?
</user>
<assistant>
<func>{"name":"check_overdue_fines", "arguments": {"book_title": "The Alchemist", "user_id": "U12345"}}</func>
</assistant>
<tool>
{"status": "success", "overdue_days": 14, "fine_amount": 7.00}
</tool>
<assistant>
"The Alchemist" is overdue by 14 days, resulting in a $7.00 fine. Would you like to pay the fine and proceed with 
borrowing "The Great Gatsby" now?
</assistant>
<user>
Yes, I'll pay the fine. Also, can I borrow "The Catcher in the Rye" next?
</user>
<assistant>
<func>{"name":"check_borrowing_limit", "arguments": {"user_id": "U12345", "additional_books": ["The Catcher in the 
Rye"]}}</func>
</assistant>
<tool>
{"status": "success", "current_borrowed": 2, "limit": 3, "available_books": ["The Catcher in the Rye"]}
</tool>
<assistant>
You currently have 2 books borrowed. "The Catcher in the Rye" is available. Would you like to borrow it now, making
your total 3 books?
</assistant>

In [38]:
dialogue=extract_dialogue(response)

In [39]:
rich.print(dialogue)

Dialogue(
    system_prompt='You are an AI assistant managing a library system. Here are the domain-specific rules:\n- Users 
can only borrow books that are currently available (status: "available").\n- Borrowing is limited to 3 books per 
user.\n- Fines apply for overdue returns (50 cents per day).\n- Users must provide valid ID to borrow books.\n- 
Book reservations are allowed only if the book is on hold.',
    conversation=[
        Message(role='user', content='I want to borrow "The Great Gatsby" and "1984".', tool_calls=None),
        Message(
            role='assistant',
            content='',
            tool_calls=[
                ToolCall(name='check_book_availability', arguments={'book_titles': ['The Great Gatsby', '1984']})
            ]
        ),
        Message(
            role='tool',
            content='{"status": "success", "availability": {"The Great Gatsby": "available", "1984": "checked 
out"}}',
            tool_calls=None
        ),
        Message(
            role='assistant',
            content='"The Great Gatsby" is available for borrowing, but "1984" is currently checked out. Would you 
like to reserve "1984" or choose a different book?',
            tool_calls=None
        ),
        Message(role='user', content='Reserve "1984" for me.', tool_calls=None),
        Message(
            role='assistant',
            content='',
            tool_calls=[ToolCall(name='reserve_book', arguments={'book_title': '1984', 'user_id': 'U12345'})]
        ),
        Message(
            role='tool',
            content='{"status": "success", "message": "Reservation request for \'1984\' has been processed. You 
will be notified when it becomes available."}',
            tool_calls=None
        ),
        Message(
            role='assistant',
            content='Your reservation for "1984" is confirmed. You can borrow up to 3 books at a time. Would you 
like to proceed with borrowing "The Great Gatsby" now?',
            tool_calls=None
        ),
        Message(
            role='user',
            content='Yes, but I need to return "The Alchemist" first. How long has it been overdue?',
            tool_calls=None
        ),
        Message(
            role='assistant',
            content='',
            tool_calls=[
                ToolCall(
                    name='check_overdue_fines',
                    arguments={'book_title': 'The Alchemist', 'user_id': 'U12345'}
                )
            ]
        ),
        Message(
            role='tool',
            content='{"status": "success", "overdue_days": 14, "fine_amount": 7.00}',
            tool_calls=None
        ),
        Message(
            role='assistant',
            content='"The Alchemist" is overdue by 14 days, resulting in a $7.00 fine. Would you like to pay the 
fine and proceed with borrowing "The Great Gatsby" now?',
            tool_calls=None
        ),
        Message(
            role='user',
            content='Yes, I\'ll pay the fine. Also, can I borrow "The Catcher in the Rye" next?',
            tool_calls=None
        ),
        Message(
            role='assistant',
            content='',
            tool_calls=[
                ToolCall(
                    name='check_borrowing_limit',
                    arguments={'user_id': 'U12345', 'additional_books': ['The Catcher in the Rye']}
                )
            ]
        ),
        Message(
            role='tool',
            content='{"status": "success", "current_borrowed": 2, "limit": 3, "available_books": ["The Catcher in 
the Rye"]}',
            tool_calls=None
        ),
        Message(
            role='assistant',
            content='You currently have 2 books borrowed. "The Catcher in the Rye" is available. Would you like to 
borrow it now, making your total 3 books?',
            tool_calls=None
        )
    ]
)

In [40]:
json_data = json.dumps(dialogue.model_dump(), ensure_ascii=False, indent=4)
rich.print(json_data)

{
    "system_prompt": "You are an AI assistant managing a library system. Here are the domain-specific rules:\n- 
Users can only borrow books that are currently available (status: \"available\").\n- Borrowing is limited to 3 
books per user.\n- Fines apply for overdue returns (50 cents per day).\n- Users must provide valid ID to borrow 
books.\n- Book reservations are allowed only if the book is on hold.",
    "conversation": [
        {
            "role": "user",
            "content": "I want to borrow \"The Great Gatsby\" and \"1984\".",
            "tool_calls": null
        },
        {
            "role": "assistant",
            "content": "",
            "tool_calls": [
                {
                    "name": "check_book_availability",
                    "arguments": {
                        "book_titles": [
                            "The Great Gatsby",
                            "1984"
                        ]
                    }
                }
            ]
        },
        {
            "role": "tool",
            "content": "{\"status\": \"success\", \"availability\": {\"The Great Gatsby\": \"available\", \"1984\":
\"checked out\"}}",
            "tool_calls": null
        },
        {
            "role": "assistant",
            "content": "\"The Great Gatsby\" is available for borrowing, but \"1984\" is currently checked out. 
Would you like to reserve \"1984\" or choose a different book?",
            "tool_calls": null
        },
        {
            "role": "user",
            "content": "Reserve \"1984\" for me.",
            "tool_calls": null
        },
        {
            "role": "assistant",
            "content": "",
            "tool_calls": [
                {
                    "name": "reserve_book",
                    "arguments": {
                        "book_title": "1984",
                        "user_id": "U12345"
                    }
                }
            ]
        },
        {
            "role": "tool",
            "content": "{\"status\": \"success\", \"message\": \"Reservation request for '1984' has been processed.
You will be notified when it becomes available.\"}",
            "tool_calls": null
        },
        {
            "role": "assistant",
            "content": "Your reservation for \"1984\" is confirmed. You can borrow up to 3 books at a time. Would 
you like to proceed with borrowing \"The Great Gatsby\" now?",
            "tool_calls": null
        },
        {
            "role": "user",
            "content": "Yes, but I need to return \"The Alchemist\" first. How long has it been overdue?",
            "tool_calls": null
        },
        {
            "role": "assistant",
            "content": "",
            "tool_calls": [
                {
                    "name": "check_overdue_fines",
                    "arguments": {
                        "book_title": "The Alchemist",
                        "user_id": "U12345"
                    }
                }
            ]
        },
        {
            "role": "tool",
            "content": "{\"status\": \"success\", \"overdue_days\": 14, \"fine_amount\": 7.00}",
            "tool_calls": null
        },
        {
            "role": "assistant",
            "content": "\"The Alchemist\" is overdue by 14 days, resulting in a $7.00 fine. Would you like to pay 
the fine and proceed with borrowing \"The Great Gatsby\" now?",
            "tool_calls": null
        },
        {
            "role": "user",
            "content": "Yes, I'll pay the fine. Also, can I borrow \"The Catcher in the Rye\" next?",
            "tool_calls": null
        },
        {
            "role": "assistant",
            "content": "",
            "tool_calls": [
                {
                    "name": "check_borrowing_limit",
                    "arguments": {
                        "user_id": "U12345",
                        "additional_books": [
                            "The Catcher in t

In [41]:
save_demo(json_data,"dialogue_demo.txt")

## trajectory_refinement

In [42]:
def extract_trajectory(response: str) -> Trajectory:
    # 1. 提取工具集 <toolsets>
    toolsets_match = re.search(r'<toolsets>(.*?)</toolsets>', response, re.DOTALL)
    toolsets_data = []
    if toolsets_match:
        try:
            toolsets_data = json.loads(toolsets_match.group(1).strip())
        except json.JSONDecodeError:
            pass # 实际开发中可增加更强的 JSON 修复逻辑

    # 2. 提取系统提示词 <system>
    system_match = re.search(r'<system>(.*?)</system>', response, re.DOTALL)
    system_prompt = system_match.group(1).strip() if system_match else ""

    # 3. 提取对话历史 (按顺序匹配所有 user, assistant, tool 标签)
    # 使用正则表达式按顺序捕获
    tags_pattern = re.compile(r'<(user|assistant|tool)>(.*?)</\1>', re.DOTALL)
    all_turns = tags_pattern.findall(response)

    conversation = []

    for role, content in all_turns:
        content = content.strip()
        tool_calls = []

        # 3. 如果是 assistant，进一步解析内部的 <func> 标签
        if role == 'assistant':
            func_pattern = re.compile(r'<func>(.*?)</func>', re.DOTALL)
            funcs = func_pattern.findall(content)
            
            for f_json in funcs:
                try:
                    # 清洗 JSON 字符串（处理模型可能多出的换行）
                    clean_json = re.sub(r'[\x00-\x1F\x7F]', '', f_json.strip())
                    f_data = json.loads(clean_json)
                    tool_calls.append(ToolCall(
                        name=f_data.get("name", ""),
                        arguments=f_data.get("arguments", {})
                    ))
                except json.JSONDecodeError:
                    continue # 或者记录解析失败
            
            # 移除 content 中的 <func> 部分，只保留纯文本回复（可选）
            content = func_pattern.sub('', content).strip()

        conversation.append(Message(
            role=role,
            content=content,
            tool_calls=tool_calls if tool_calls else None
        ))

    return Trajectory(
        toolsets=[ToolDefinition(**t) for t in toolsets_data],
        system_prompt=system_prompt,
        conversation=conversation
    )

In [43]:
trajectory_refinement_prompt=""
with open("../../src/gem/prompts/trajectory_refinement.md","r") as f:
    trajectory_refinement_prompt=f.read()

In [44]:
def trajectory_refinement(workflow:Workflow,dialogue:Dialogue)->str:
    tools:list[dict]=workflow.tools
    tools_str=json.dumps(tools,ensure_ascii=False)
    prompt=(
        trajectory_refinement_prompt
        .replace("{tools}",tools_str)
        .replace("{our_traj}",json.dumps(dialogue.model_dump(),ensure_ascii=False))
    )
    return call_llm(prompt)

In [45]:
response=trajectory_refinement(wls[0],dialogue)

In [46]:
rich.print(response)

<toolsets>
[
    {
        "name": "check_book_availability",
        "description": "Check if books are available for borrowing",
        "parameters": {
            "type": "object",
            "properties": {
                "book_titles": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "List of book titles to check"
                },
                "user_id": {
                    "type": "string",
                    "description": "User ID for borrowing eligibility check"
                }
            },
            "required": ["book_titles"]
        }
    },
    {
        "name": "reserve_book",
        "description": "Reserve a book for future borrowing",
        "parameters": {
            "type": "object",
            "properties": {
                "book_title": {"type": "string"},
                "user_id": {"type": "string"},
                "reservation_date": {"type": "string", "format": "date-time"}
            },
            "required": ["book_title", "user_id"]
        }
    },
    {
        "name": "check_overdue_fines",
        "description": "Calculate fines for overdue books",
        "parameters": {
            "type": "object",
            "properties": {
                "book_title": {"type": "string"},
                "user_id": {"type": "string"},
                "return_date": {"type": "string", "format": "date"}
            },
            "required": ["book_title", "user_id"]
        }
    },
    {
        "name": "pay_fine",
        "description": "Process payment for overdue fines",
        "parameters": {
            "type": "object",
            "properties": {
                "book_title": {"type": "string"},
                "user_id": {"type": "string"},
                "payment_amount": {"type": "number"}
            },
            "required": ["book_title", "user_id"]
        }
    },
    {
        "name": "check_borrowing_limit",
        "description": "Check current borrowing limit status",
        "parameters": {
            "type": "object",
            "properties": {
                "user_id": {"type": "string"},
                "additional_books": {
                    "type": "array",
                    "items": {"type": "string"}
                }
            },
            "required": ["user_id"]
        }
    },
    {
        "name": "check_reservation_status",
        "description": "Check book reservation availability",
        "parameters": {
            "type": "object",
            "properties": {
                "book_title": {"type": "string"},
                "user_id": {"type": "string"}
            },
            "required": ["book_title"]
        }
    }
]
</toolsets>

<system>
You are an AI assistant managing a library system. Database schema:
- Users: {user_id, name, email, borrowed_books: , reservations: , fine_balance: number}
- Books: {book_id, title, status: "available"/"checked out"/"on hold", due_date: date}
- Reservations: {reservation_id, user_id, book_id, request_date: date}

Rules:
1. Users can only borrow books with status "available"
2. Borrowing limit: 3 books per user
3. Fines: 50 cents/day for overdue books (calculated from due_date)
4. Reservations require book to be "on hold" and user must have 2 or fewer borrowed books
5. Users must provide valid ID (user_id) for all actions
6. Reservations are processed only if book is "on hold" and not already reserved by user
</system>

<user>
I want to borrow "The Great Gatsby" and "1984".
</user>
<assistant>
<func>
{"name": "check_book_availability", "arguments": {"book_titles": ["The Great Gatsby", "1984"], "user_id": 
"U12345"}}
</func>
</assistant>
<tool>
{"status": "success", "availability": {"The Great Gatsby": "available", "1984": "checked out"}, "borrowing_limit": 
3}
</tool>
<assistant>
"The Great Gatsby" is available, but "1984" is currently checked out. Would you like to reserve "1984" or choose a 
different book? Also, yo

In [47]:
trajectory=extract_trajectory(response)

In [48]:
rich.print(trajectory)

Trajectory(
    toolsets=[
        ToolDefinition(
            name='check_book_availability',
            description='Check if books are available for borrowing',
            parameters={
                'type': 'object',
                'properties': {
                    'book_titles': {
                        'type': 'array',
                        'items': {'type': 'string'},
                        'description': 'List of book titles to check'
                    },
                    'user_id': {'type': 'string', 'description': 'User ID for borrowing eligibility check'}
                },
                'required': ['book_titles']
            }
        ),
        ToolDefinition(
            name='reserve_book',
            description='Reserve a book for future borrowing',
            parameters={
                'type': 'object',
                'properties': {
                    'book_title': {'type': 'string'},
                    'user_id': {'type': 'string'},
                    'reservation_date': {'type': 'string', 'format': 'date-time'}
                },
                'required': ['book_title', 'user_id']
            }
        ),
        ToolDefinition(
            name='check_overdue_fines',
            description='Calculate fines for overdue books',
            parameters={
                'type': 'object',
                'properties': {
                    'book_title': {'type': 'string'},
                    'user_id': {'type': 'string'},
                    'return_date': {'type': 'string', 'format': 'date'}
                },
                'required': ['book_title', 'user_id']
            }
        ),
        ToolDefinition(
            name='pay_fine',
            description='Process payment for overdue fines',
            parameters={
                'type': 'object',
                'properties': {
                    'book_title': {'type': 'string'},
                    'user_id': {'type': 'string'},
                    'payment_amount': {'type': 'number'}
                },
                'required': ['book_title', 'user_id']
            }
        ),
        ToolDefinition(
            name='check_borrowing_limit',
            description='Check current borrowing limit status',
            parameters={
                'type': 'object',
                'properties': {
                    'user_id': {'type': 'string'},
                    'additional_books': {'type': 'array', 'items': {'type': 'string'}}
                },
                'required': ['user_id']
            }
        ),
        ToolDefinition(
            name='check_reservation_status',
            description='Check book reservation availability',
            parameters={
                'type': 'object',
                'properties': {'book_title': {'type': 'string'}, 'user_id': {'type': 'string'}},
                'required': ['book_title']
            }
        )
    ],
    system_prompt='You are an AI assistant managing a library system. Database schema:\n- Users: {user_id, name, 
email, borrowed_books: [book_id], reservations: [book_id], fine_balance: number}\n- Books: {book_id, title, status:
"available"/"checked out"/"on hold", due_date: date}\n- Reservations: {reservation_id, user_id, book_id, 
request_date: date}\n\nRules:\n1. Users can only borrow books with status "available"\n2. Borrowing limit: 3 books 
per user\n3. Fines: 50 cents/day for overdue books (calculated from due_date)\n4. Reservations require book to be 
"on hold" and user must have 2 or fewer borrowed books\n5. Users must provide valid ID (user_id) for all 
actions\n6. Reservations are processed only if book is "on hold" and not already reserved by user',
    conversation=[
        Message(role='user', content='I want to borrow "The Great Gatsby" and "1984".', tool_calls=None),
        Message(
            role='assistant',
            content='',
            tool_calls=[
                ToolCall(
                    name='check_book_availabili

In [49]:
json_data = json.dumps(trajectory.model_dump(), ensure_ascii=False, indent=4)
rich.print(json_data)

{
    "toolsets": [
        {
            "name": "check_book_availability",
            "description": "Check if books are available for borrowing",
            "parameters": {
                "type": "object",
                "properties": {
                    "book_titles": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        },
                        "description": "List of book titles to check"
                    },
                    "user_id": {
                        "type": "string",
                        "description": "User ID for borrowing eligibility check"
                    }
                },
                "required": [
                    "book_titles"
                ]
            }
        },
        {
            "name": "reserve_book",
            "description": "Reserve a book for future borrowing",
            "parameters": {
                "type": "object",
                "properties": {
                    "book_title": {
                        "type": "string"
                    },
                    "user_id": {
                        "type": "string"
                    },
                    "reservation_date": {
                        "type": "string",
                        "format": "date-time"
                    }
                },
                "required": [
                    "book_title",
                    "user_id"
                ]
            }
        },
        {
            "name": "check_overdue_fines",
            "description": "Calculate fines for overdue books",
            "parameters": {
                "type": "object",
                "properties": {
                    "book_title": {
                        "type": "string"
                    },
                    "user_id": {
                        "type": "string"
                    },
                    "return_date": {
                        "type": "string",
                        "format": "date"
                    }
                },
                "required": [
                    "book_title",
                    "user_id"
                ]
            }
        },
        {
            "name": "pay_fine",
            "description": "Process payment for overdue fines",
            "parameters": {
                "type": "object",
                "properties": {
                    "book_title": {
                        "type": "string"
                    },
                    "user_id": {
                        "type": "string"
                    },
                    "payment_amount": {
                        "type": "number"
                    }
                },
                "required": [
                    "book_title",
                    "user_id"
                ]
            }
        },
        {
            "name": "check_borrowing_limit",
            "description": "Check current borrowing limit status",
            "parameters": {
                "type": "object",
                "properties": {
                    "user_id": {
                        "type": "string"
                    },
                    "additional_books": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    }
                },
                "required": [
                    "user_id"
                ]
            }
        },
        {
            "name": "check_reservation_status",
            "description": "Check book reservation availability",
            "parameters": {
                "type": "object",
                "properties": {
                    "book_title": {
                        "type": "string"
                    },
                    "user_id": {
                        "type": "string"
                    }
                },
                "requi

In [50]:
save_demo(json_data,"trajectory_demo.txt")

## hallucination_detection

In [51]:
from pydantic import BaseModel, Field, field_validator

class EvaluationResult(BaseModel):
    """轨迹评估结果类"""
    R1: int = Field(description="工具调用幻觉得分 (0 或 1)")
    R2: int = Field(description="能力幻觉得分 (0 或 1)")
    R3: int = Field(description="上下文幻觉得分 (0 或 1)")

    @field_validator('R1', 'R2', 'R3')
    @classmethod
    def check_binary(cls, v: int) -> int:
        """确保得分只能是 0 或 1"""
        if v not in (0, 1):
            raise ValueError("Score must be 0 or 1")
        return v

In [52]:
import re
import json
from typing import Optional

def extract_evaluation(response: str) -> Optional[EvaluationResult]:
    """
    从模型响应中提取 JSON 评估结果。
    支持处理带有 Markdown 标签的 JSON 或纯 JSON 字符串。
    """
    # 1. 尝试寻找最外层的 JSON 花括号
    # 这样即使模型输出了 "Here is the result: { ... }" 也能成功提取
    json_pattern = re.compile(r'(\{.*?\})', re.DOTALL)
    match = json_pattern.search(response)
    
    if not match:
        print("Error: No JSON object found in response.")
        return None
    
    try:
        # 2. 清理可能存在的不可见字符并解析
        clean_json_str = re.sub(r'[\x00-\x1F\x7F]', '', match.group(1).strip())
        data = json.loads(clean_json_str)
        
        # 3. 实例化 Pydantic 模型（自动执行校验）
        return EvaluationResult.model_validate(data)
        
    except (json.JSONDecodeError, ValueError) as e:
        print(f"Extraction Failed: {e}")
        return None

In [53]:
hallucination_detection_prompt=""
with open("../../src/gem/prompts/hallucination_detection.md","r") as f:
    hallucination_detection_prompt=f.read()

In [54]:
def hallucination_detection(trajectory:Trajectory)->str:
    prompt=(
        hallucination_detection_prompt
        .replace("{trajectory}",trajectory.model_dump_json())
    )
    return call_llm(prompt)

In [55]:
response=hallucination_detection(trajectory)

In [56]:
rich.print(response)

{
  "R1": 1,
  "R2": 1,
  "R3": 1
}

In [57]:
evaluation=extract_evaluation(response)

In [58]:
rich.print(evaluation)

EvaluationResult(R1=1, R2=1, R3=1)

In [59]:
json_data=json.dumps(evaluation.model_dump(),ensure_ascii=False,indent=4)

In [60]:
save_demo(json_data,"evaluation_demo.txt")

In [61]:
# TODO: 只有R1,R2,R3都为1的数据，才被保留下来，等待后续转换格式，用于强化学习或监督微调

In [62]:
# TODO: 把保存下来的数据转为openai messages格式
# 转换的逻辑参考hardtry/utils/convert_hardgen_to_messages.py
# 最后的数据只包含system|user|assistant role